# 03 — 条件C/D用official6特徴cacheの作成

独立したWSL環境に `requirements-official.txt` を導入し、そのkernelを選択してください。
取得済み音声の監査 → official6 manifest生成 → 公式モデル一致確認を行い、本番セルでは
全dataset preflight → encoder/parity照合 → 全dataset smoke → 未commit断片回収 → Large特徴抽出
の順序を固定します。

後半Notebookへの受け渡し契約は `official6 manifest + Large feature cache + parity.json` です。
既存の `runs/official_cd` 以下を使うため、検証済みartifactはそのまま再利用できます。
全実行フラグは初期値Falseです。分類器の学習・評価はこのNotebookに含みません。


In [ ]:
from pathlib import Path
import os, sys, json
from collections import Counter

ROOT = Path.cwd().resolve()
if not (ROOT / 'ser_pipeline').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'ser_pipeline').is_dir():
    raise RuntimeError('リポジトリまたはnotebooksディレクトリから実行してください。')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ser_pipeline.cli import main
from ser_pipeline.official import load_official_head, OfficialEmotion2vecEncoder, require_parity_report
from ser_pipeline.features import extract_feature_cache
from ser_pipeline.manifest import (
    audit_dataset, build_manifest, generate_msp_audio_duplicate_audit,
    generate_msp_missing_audio_exclusion_contract, load_manifest, validate_manifest,
)
from ser_pipeline.duplicates import (
    generate_msp_audio_duplicate_exclusion_contract,
    load_msp_audio_duplicate_audit,
    load_msp_audio_duplicate_exclusion_contract,
)
from ser_pipeline.cache import cleanup_uncommitted_cache_fragments, _atomic_json
from ser_pipeline.contracts import label_profile_for_mapping_version
from ser_pipeline.preflight import preflight_feature_extraction, smoke_test_feature_extraction
from ser_pipeline.readers import resolved_dataset_root

REVISION = '6c303ba987b86b93193de93e34bb2b077a6bedc4'
HF_HOME = Path('/mnt/c/Users/RD004/.cache/huggingface/hub') if sys.platform != 'win32' else Path.home() / '.cache/huggingface/hub'
SNAPSHOT = Path(os.environ.get('SER_OFFICIAL_SNAPSHOT', str(HF_HOME / 'models--emotion2vec--emotion2vec_plus_large' / 'snapshots' / REVISION)))
OUTPUT = ROOT / 'runs' / 'official_cd'
PARITY_REPORT = OUTPUT / 'parity.json'
PREFLIGHT_REPORT = OUTPUT / 'feature_preflight.json'
SMOKE_REPORT = OUTPUT / 'feature_smoke.json'
EXTRACTION_REPORT = OUTPUT / 'feature_extraction.json'
DEVICE = 'cpu'

# `build-manifest --label-profile official6` で作る承認済みmanifestと音声rootです。
# ここでは文字列を設定するだけで、各実行フラグをTrueにするまでデータは参照しません。
DATA_ROOT = (
    Path('/mnt/c/Users/RD004/Documents/lab/data')
    if sys.platform != 'win32'
    else Path(r'C:\Users\RD004\Documents\lab\data')
)
MANIFEST_DIR = ROOT / 'runs' / 'ser_manifests'
MANIFESTS = {
    'msp_podcast': MANIFEST_DIR / 'msp_podcast_official6_v1.jsonl',
    'hcudb1': MANIFEST_DIR / 'hcudb1_official6_v1.jsonl',
}
AUDIO_ROOTS = {
    'msp_podcast': DATA_ROOT / 'MSP_PODCAST',
    'hcudb1': DATA_ROOT / 'HCUDB1',
}
MSP_MISSING_CONTRACT = MANIFEST_DIR / 'msp_missing_audio_exclusions_official6_v1.json'
MSP_DUPLICATE_AUDIT = MANIFEST_DIR / 'msp_audio_duplicate_audit_official6_v1.json'
MSP_DUPLICATE_CANDIDATES = MANIFEST_DIR / 'msp_audio_duplicate_candidates_official6_v1.csv'
MSP_DUPLICATE_EXCLUSION_CONTRACT = MANIFEST_DIR / 'msp_audio_duplicate_exclusions_official6_v1.json'
MSP_PRIOR_DUPLICATE_AUDIT = MANIFEST_DIR / 'msp_audio_duplicate_audit_v1.json'
MSP_PRIOR_DUPLICATE_EXCLUSION_CONTRACT = MANIFEST_DIR / 'msp_audio_duplicate_exclusions_v1.json'
MSP_PRIOR_DUPLICATE_EXCLUSION_SHA256 = 'cc1be85082eb75d4e2068551454988fe41e3b58b8a22ade3bd4fc86a2e33f888'

# 既存4クラス契約の41件は検証して自動継承します。ここにはofficial6で新しく承認する21件だけを記録します。
MSP_EXPECTED_MISSING_SHA256 = None
MSP_NEW_OFFICIAL6_DUPLICATE_EXCLUDE_IDS = [
    # ラベル不一致9グループは、従来方針どおり両方を除外する。
    'MSP-PODCAST_0103_0504',
    'MSP-PODCAST_0103_0505',
    'MSP-PODCAST_0103_0608',
    'MSP-PODCAST_0103_0609',
    'MSP-PODCAST_0103_0658',
    'MSP-PODCAST_0103_0659',
    'MSP-PODCAST_0103_0673',
    'MSP-PODCAST_0103_0674',
    'MSP-PODCAST_0125_0027',
    'MSP-PODCAST_0125_0031',
    'MSP-PODCAST_0566_0097',
    'MSP-PODCAST_0581_0097',
    'MSP-PODCAST_0566_0100',
    'MSP-PODCAST_0581_0100',
    'MSP-PODCAST_0673_0549',
    'MSP-PODCAST_0680_0549',
    'MSP-PODCAST_2197_0002',
    'MSP-PODCAST_2200_0014',
    # 同一ラベル3グループは一方を残し、従来と同じ順序規則で後者を除外する。
    'MSP-PODCAST_0581_0095',
    'MSP-PODCAST_0581_0438',
    'MSP-PODCAST_0680_0125_0002',
]
MSP_EXPECTED_DUPLICATE_EXCLUSION_SHA256 = None

CACHES = {dataset: OUTPUT / 'cache' / dataset for dataset in MANIFESTS}
OFFICIAL_ARTIFACT_CONTRACT = {
    'manifests': MANIFESTS,
    'caches': CACHES,
    'parity_report': PARITY_REPORT,
}

RUN_SOURCE_AUDIT = True 
RUN_GENERATE_MSP_MISSING_CONTRACT = True
RUN_GENERATE_MSP_DUPLICATE_AUDIT = True
RUN_GENERATE_MSP_DUPLICATE_EXCLUSION_CONTRACT = True
RUN_BUILD_HCUDB_MANIFEST = True
RUN_BUILD_MSP_MANIFEST = True
RUN_AUDIT = True
RUN_PARITY = True
RUN_PREFLIGHT = True
RUN_FULL_EXTRACTION = True

def require_manifests():
    resolved = {dataset: Path(path) for dataset, path in MANIFESTS.items()}
    missing = [str(path) for path in resolved.values() if not path.is_file()]
    if missing:
        raise FileNotFoundError('official6 manifestが未生成です。先にmanifest生成セルを実行してください: ' + ', '.join(missing))
    return resolved

def require_approval_sha(value, label):
    normalized = str(value or '').strip().lower()
    if len(normalized) != 64 or any(character not in '0123456789abcdef' for character in normalized):
        raise ValueError(f'{label}へ、直前の生成結果を確認して表示された64桁SHA-256を設定してください。')
    return normalized

def approved_official6_duplicate_exclude_ids():
    prior_audit, _ = load_msp_audio_duplicate_audit(MSP_PRIOR_DUPLICATE_AUDIT)
    prior_contract, _ = load_msp_audio_duplicate_exclusion_contract(
        MSP_PRIOR_DUPLICATE_EXCLUSION_CONTRACT,
        prior_audit,
        expected_sha256=MSP_PRIOR_DUPLICATE_EXCLUSION_SHA256,
    )
    current_audit, _ = load_msp_audio_duplicate_audit(MSP_DUPLICATE_AUDIT)
    current_by_id = {str(record['utterance_id']): record for record in current_audit['records']}
    current_group_by_id = {
        str(identifier): str(group['group_id'])
        for group in current_audit['duplicate_groups']
        for identifier in group['member_ids']
    }
    identity_fields = (
        'audio_relpath', 'source_split', 'split', 'speaker_id', 'original_emotion',
        'byte_sha256', 'decoded_waveform_sha256',
    )
    inherited_ids = []
    for prior_record in prior_contract['records']:
        identifier = str(prior_record['utterance_id'])
        current_record = current_by_id.get(identifier)
        if current_record is None or any(current_record[field] != prior_record[field] for field in identity_fields):
            raise ValueError(f'既存重複除外IDをofficial6監査へ安全に継承できません: {identifier}')
        if current_group_by_id.get(identifier) != prior_record['duplicate_group_id']:
            raise ValueError(f'既存重複除外IDのduplicate groupが変化しています: {identifier}')
        inherited_ids.append(identifier)
    new_ids = [str(identifier) for identifier in MSP_NEW_OFFICIAL6_DUPLICATE_EXCLUDE_IDS]
    if len(inherited_ids) != 41 or len(new_ids) != 21:
        raise ValueError('重複除外の承認件数が想定（既存41件 + official6新規21件）と一致しません。')
    combined = inherited_ids + new_ids
    if len(combined) != len(set(combined)):
        raise ValueError('既存契約とofficial6新規承認IDに重複があります。')
    unknown = sorted(set(combined) - set(current_group_by_id))
    if unknown:
        raise ValueError(f'official6重複監査の候補にない承認IDがあります: {unknown[:5]}')
    print(f'重複除外ID: 既存契約から{len(inherited_ids)}件を継承 + official6新規{len(new_ids)}件 = {len(combined)}件')
    return combined


## 1. 取得済み音声の監査とofficial6 manifest生成

以下のセルは上から順番に実行します。設定セルのフラグは一度に1つだけTrueにしてください。

1. `RUN_SOURCE_AUDIT`で取得済み音声の状況を確認する。F/Uが全件欠損の状態では先へ進めない。
2. MSP欠損契約を生成し、内容確認後に表示SHA-256を設定セルへ転記する。
3. MSP重複監査を生成し、候補CSVを人手で確認する。
4. 既存4クラス契約の41件を検証・継承し、official6新規21件と結合して重複除外契約を生成する。
   表示されたSHA-256を設定セルへ転記する。
5. HCUDBとMSPのofficial6 manifestを生成する。
6. `RUN_AUDIT`で完成したmanifestを検証する。

生成物が既に存在する場合は上書きしません。再生成する場合は、古い承認済みartifactを別途整理してから新しい出力先を指定してください。


In [ ]:
path_status = {
    'audio_roots': {dataset: {'path': str(path), 'exists': path.is_dir()} for dataset, path in AUDIO_ROOTS.items()},
    'manifests': {dataset: {'path': str(path), 'exists': path.is_file()} for dataset, path in MANIFESTS.items()},
    'msp_contracts': {
        'missing': {'path': str(MSP_MISSING_CONTRACT), 'exists': MSP_MISSING_CONTRACT.is_file()},
        'duplicate_audit': {'path': str(MSP_DUPLICATE_AUDIT), 'exists': MSP_DUPLICATE_AUDIT.is_file()},
        'duplicate_candidates': {'path': str(MSP_DUPLICATE_CANDIDATES), 'exists': MSP_DUPLICATE_CANDIDATES.is_file()},
        'duplicate_exclusions': {'path': str(MSP_DUPLICATE_EXCLUSION_CONTRACT), 'exists': MSP_DUPLICATE_EXCLUSION_CONTRACT.is_file()},
    },
}
print(json.dumps(path_status, ensure_ascii=False, indent=2))


In [ ]:
if RUN_SOURCE_AUDIT:
    OUTPUT.mkdir(parents=True, exist_ok=True)
    for dataset, root in AUDIO_ROOTS.items():
        report = audit_dataset(dataset, root, label_profile='official6')
        _atomic_json(report, OUTPUT / f'{dataset}_official6_source_audit.json')
        print(dataset, json.dumps(report, ensure_ascii=False, indent=2))
else:
    print('official6元データ監査は未実行です。')


In [ ]:
if RUN_GENERATE_MSP_MISSING_CONTRACT:
    if MSP_MISSING_CONTRACT.exists():
        raise FileExistsError(f'既存契約を上書きしません: {MSP_MISSING_CONTRACT}')
    report = generate_msp_missing_audio_exclusion_contract(
        AUDIO_ROOTS['msp_podcast'], MSP_MISSING_CONTRACT, label_profile='official6',
    )
    print(json.dumps(report, ensure_ascii=False, indent=2))
    print('内容を確認後、normalized_sha256をMSP_EXPECTED_MISSING_SHA256へ転記してください。')
else:
    print('MSP official6欠損契約は未生成です。')


In [ ]:
if RUN_GENERATE_MSP_DUPLICATE_AUDIT:
    approved_missing_sha = require_approval_sha(MSP_EXPECTED_MISSING_SHA256, 'MSP_EXPECTED_MISSING_SHA256')
    if MSP_DUPLICATE_AUDIT.exists() or MSP_DUPLICATE_CANDIDATES.exists():
        raise FileExistsError('既存のofficial6重複監査artifactを上書きしません。')
    report = generate_msp_audio_duplicate_audit(
        AUDIO_ROOTS['msp_podcast'],
        MSP_DUPLICATE_AUDIT,
        MSP_DUPLICATE_CANDIDATES,
        approved_missing_audio_exclusion_contract=MSP_MISSING_CONTRACT,
        expected_missing_audio_exclusion_sha256=approved_missing_sha,
        label_profile='official6',
    )
    print(json.dumps(report, ensure_ascii=False, indent=2))
    print('候補CSVを確認してください:', MSP_DUPLICATE_CANDIDATES)
else:
    print('MSP official6重複監査は未実行です。')


In [ ]:
if RUN_GENERATE_MSP_DUPLICATE_EXCLUSION_CONTRACT:
    if MSP_DUPLICATE_EXCLUSION_CONTRACT.exists():
        raise FileExistsError(f'既存契約を上書きしません: {MSP_DUPLICATE_EXCLUSION_CONTRACT}')
    approved_duplicate_ids = approved_official6_duplicate_exclude_ids()
    report = generate_msp_audio_duplicate_exclusion_contract(
        MSP_DUPLICATE_AUDIT,
        approved_duplicate_ids,
        MSP_DUPLICATE_EXCLUSION_CONTRACT,
    )
    print(json.dumps(report, ensure_ascii=False, indent=2))
    print('内容を確認後、normalized_sha256をMSP_EXPECTED_DUPLICATE_EXCLUSION_SHA256へ転記してください。')
else:
    print('MSP official6重複除外契約は未生成です。候補CSV確認後に実行します。')


In [ ]:
if RUN_BUILD_HCUDB_MANIFEST:
    output = MANIFESTS['hcudb1']
    if output.exists():
        raise FileExistsError(f'既存manifestを上書きしません: {output}')
    report = build_manifest(
        'hcudb1', AUDIO_ROOTS['hcudb1'], output,
        strict=True, inspect_excluded_audio=False, label_profile='official6',
    )
    _atomic_json(report, OUTPUT / 'hcudb1_official6_manifest_build.json')
    print(json.dumps(report, ensure_ascii=False, indent=2))
else:
    print('HCUDB official6 manifestは未生成です。')


In [ ]:
if RUN_BUILD_MSP_MANIFEST:
    approved_missing_sha = require_approval_sha(MSP_EXPECTED_MISSING_SHA256, 'MSP_EXPECTED_MISSING_SHA256')
    approved_duplicate_sha = require_approval_sha(
        MSP_EXPECTED_DUPLICATE_EXCLUSION_SHA256,
        'MSP_EXPECTED_DUPLICATE_EXCLUSION_SHA256',
    )
    output = MANIFESTS['msp_podcast']
    if output.exists():
        raise FileExistsError(f'既存manifestを上書きしません: {output}')
    report = build_manifest(
        'msp_podcast', AUDIO_ROOTS['msp_podcast'], output,
        strict=True,
        inspect_excluded_audio=False,
        approved_exclusion_contract=MSP_MISSING_CONTRACT,
        expected_exclusion_sha256=approved_missing_sha,
        duplicate_audit=MSP_DUPLICATE_AUDIT,
        approved_duplicate_exclusion_contract=MSP_DUPLICATE_EXCLUSION_CONTRACT,
        expected_duplicate_exclusion_sha256=approved_duplicate_sha,
        label_profile='official6',
    )
    _atomic_json(report, OUTPUT / 'msp_podcast_official6_manifest_build.json')
    print(json.dumps(report, ensure_ascii=False, indent=2))
else:
    print('MSP official6 manifestは未生成です。')


In [ ]:
if RUN_AUDIT:
    for dataset, manifest_path in require_manifests().items():
        if AUDIO_ROOTS[dataset] is None:
            raise ValueError('AUDIO_ROOTSを設定してください。')
        resolved_root = resolved_dataset_root(dataset, AUDIO_ROOTS[dataset]).resolve()
        report = validate_manifest(manifest_path, audio_root=resolved_root, audio_root_resolved=True)
        rows = load_manifest(manifest_path)
        profiles = {label_profile_for_mapping_version(r['mapping_version']) for r in rows}
        if profiles != {'official6'}:
            raise ValueError(f'{dataset} manifestはofficial6ラベル契約ではありません。')
        counts = Counter((r['split'], r['original_emotion'], r['mapped_emotion'], r['approximate_mapping']) for r in rows if r['included'])
        print(dataset, '元感情・統合先・近似フラグ別の件数:', counts)
        print('除外:', Counter(reason for r in rows if not r['included'] for reason in r['exclusion_reasons']))
        _atomic_json(report, OUTPUT / f'{dataset}_manifest_validation.json')
else:
    print('データ監査は未実行です。')


In [ ]:
if RUN_PARITY:
    main(['verify-official', '--snapshot', str(SNAPSHOT), '--audio', str(SNAPSHOT / 'example/test.wav'),
          '--output', str(PARITY_REPORT), '--device', DEVICE])
else:
    print('1音声での公式一致・時間・メモリ・保存量の検証は未実行です。')


In [ ]:
if RUN_PREFLIGHT:
    manifests = require_manifests()
    for dataset in manifests:
        print(f'[PRECHECK] {dataset} start')
    preflight_report = preflight_feature_extraction(
        manifests, AUDIO_ROOTS, CACHES, PARITY_REPORT,
        expected_dim=1024, max_shard_frames=65536,
        capacity_path=OUTPUT, report_path=PREFLIGHT_REPORT,
    )
    for dataset, item in preflight_report['datasets'].items():
        print(f'[PRECHECK] {dataset} complete', json.dumps(item, ensure_ascii=False, indent=2))
else:
    print('全dataset preflightは未実行です。音声全件・resume状態・残容量をまとめて検証します。')


In [ ]:
if RUN_FULL_EXTRACTION:
    manifests = require_manifests()
    for dataset in manifests:
        print(f'[PRECHECK] {dataset} start')
    preflight_report = preflight_feature_extraction(
        manifests, AUDIO_ROOTS, CACHES, PARITY_REPORT,
        expected_dim=1024, max_shard_frames=65536,
        capacity_path=OUTPUT, report_path=PREFLIGHT_REPORT,
    )
    resolved_audio_roots = {
        dataset: Path(preflight_report['datasets'][dataset]['resolved_audio_root'])
        for dataset in manifests
    }
    for dataset in manifests:
        print(f'[PRECHECK] {dataset} complete')

    encoder = OfficialEmotion2vecEncoder(SNAPSHOT, device=DEVICE)
    report = require_parity_report(encoder.head, PARITY_REPORT)
    if report['extraction'] != encoder.provenance or report['device'] != str(encoder.device):
        raise ValueError('現在の環境で公式一致検証を再実行してください。')

    smoke_reports = {}
    for dataset, manifest_path in manifests.items():
        print(f'[SMOKE] {dataset} start')
        smoke_reports[dataset] = smoke_test_feature_extraction(
            manifest_path, resolved_audio_roots[dataset], CACHES[dataset], encoder,
            dataset=dataset, sample_size=10, expected_dim=1024, max_shard_frames=65536,
        )
        print(f'[SMOKE] {dataset} complete')
    _atomic_json({'status': 'ok', 'datasets': smoke_reports}, SMOKE_REPORT)

    recovered = {}
    for dataset in manifests:
        resume = preflight_report['datasets'][dataset]['resume']
        recovered[dataset] = cleanup_uncommitted_cache_fragments(
            CACHES[dataset], resume['recoverable_fragments'],
        )
        print(f"[RESUME] {dataset} committed={resume['committed_utterances']} "
              f"pending={resume['pending_utterances']} recovered={len(recovered[dataset])}")

    extraction_reports = {}
    for dataset, manifest_path in manifests.items():
        resume = preflight_report['datasets'][dataset]['resume']
        if resume['complete']:
            extraction_reports[dataset] = {**resume, 'action': 'validated/skip'}
            print(f'[EXTRACT] {dataset} validated/skip')
            continue
        print(f'[EXTRACT] {dataset} start')
        extraction_reports[dataset] = extract_feature_cache(
            manifest_path, resolved_audio_roots[dataset], CACHES[dataset], encoder,
            expected_dim=1024, max_shard_frames=65536,
        )
        print(f'[EXTRACT] {dataset} complete')
    _atomic_json(
        {'status': 'ok', 'datasets': extraction_reports, 'recovered_fragments': recovered},
        EXTRACTION_REPORT,
    )
    print(json.dumps(extraction_reports, ensure_ascii=False, indent=2))
    del encoder
else:
    print('全量抽出は未実行です。C/D専用official6 manifestに結び付くLarge cacheを使用します。')


## 2. 受け渡しartifactの抽出結果

後半Notebookへ渡す3点は、両データセットのofficial6 manifest、対応するLarge特徴cache、
および `runs/official_cd/parity.json` です。抽出関数が返した最終検証reportを保存・表示します。
完成済みcacheはpreflightで同じ検証を完了し、`validated/skip`として記録します。


In [ ]:
if EXTRACTION_REPORT.is_file():
    print(EXTRACTION_REPORT.read_text(encoding='utf-8'))
    print('受け渡しartifact:', OFFICIAL_ARTIFACT_CONTRACT)
else:
    print('抽出reportはまだありません。RUN_FULL_EXTRACTION=Trueでpreflightから一括実行してください。')
